# 02 — Data quality and preparation

Validate the downloaded observations, document missingness and duplicates, and produce a clean analysis table. No missing-value imputation is performed.

In [ ]:
from pathlib import Path
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/"README.md").exists() and (ROOT.parent/"README.md").exists(): ROOT=ROOT.parent
RAW=ROOT/"Dataset"/"Raw_Data"; OUT=ROOT/"Dataset"/"Processed_Data"; FIG=ROOT/"Dataset"/"Notebooks"/"Visualisations"
RAW.mkdir(parents=True,exist_ok=True); OUT.mkdir(parents=True,exist_ok=True); FIG.mkdir(parents=True,exist_ok=True)
p=RAW/"india_health_financing_world_bank.csv"
if not p.exists(): raise FileNotFoundError("Run 01_Data_Acquisition.ipynb first.")
df=pd.read_csv(p); df["year"]=pd.to_numeric(df.year,errors="coerce"); df["value"]=pd.to_numeric(df.value,errors="coerce")
summary={"rows":len(df),"duplicate_indicator_year":int(df.duplicated(["indicator_code","year"]).sum()),"missing_values":int(df.value.isna().sum()),"indicators":int(df.indicator_code.nunique())}
assert df.year.notna().all(); assert df.indicator_code.notna().all(); assert summary["duplicate_indicator_year"]==0
clean=df.dropna(subset=["value"]).sort_values(["indicator_code","year"]).copy()
clean.to_csv(OUT/"india_health_financing_clean.csv",index=False); pd.DataFrame([summary]).to_csv(OUT/"data_quality_summary.csv",index=False)
display(pd.DataFrame([summary])); display(clean.groupby("indicator").value.agg(["count","min","median","max"]))

## Interpretation safeguards
Nominal per-capita US-dollar values are affected by prices and exchange rates. OOP share measures the health-financing mix, not the share of households facing catastrophic expenditure. Always report units and year coverage.